In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# データ読み込み
DATA_PATH = "/kaggle/input/nlp-getting-started"
train = pd.read_csv(f"{DATA_PATH}/train.csv")
test  = pd.read_csv(f"{DATA_PATH}/test.csv")

print(train.shape, test.shape)
train.head()

In [ ]:
# 2. クリーニングに必要なライブラリをここで
import re, string, warnings
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS


URL_PAT  = r"http\S+|www\.\S+"
MENTION  = r"@\w+"
HTML_TAG = r"<.*?>"
PUNCT_TR = str.maketrans("", "", string.punctuation)

def clean_text(text:str) -> str:
    """小文字化→URL/メンション/HTML 削除→句読点削除→ストップワード除去"""
    text = re.sub(URL_PAT, " ", str(text))
    text = re.sub(MENTION, " ", text)
    text = re.sub(HTML_TAG, " ", text)
    text = text.translate(PUNCT_TR).lower()
    tokens = [w for w in text.split() if w and w not in ENGLISH_STOP_WORDS]
    return " ".join(tokens)

for df in (train, test):
    df[["keyword", "location"]] = df[["keyword", "location"]].fillna("")
    df["clean_text"] = df["text"].apply(clean_text)


In [ ]:
# 3. メタ列作成に emoji を使うのでここで import
import emoji

def add_meta(df):
    df["has_url"]      = df["text"].str.contains(URL_PAT).astype(int)
    df["has_keyword"]  = (df["keyword"]  != "").astype(int)
    df["has_mention"]  = df["text"].str.contains(MENTION).astype(int)
    df["has_hashtag"]  = df["text"].str.contains(r"#\w+").astype(int)
    df["has_location"] = (df["location"] != "").astype(int)
    df["text_len"]     = df["clean_text"].str.len()
    df["word_cnt"]     = df["clean_text"].str.split().apply(len)
    df["emoji_cnt"]    = df["text"].apply(
                           lambda x: sum(ch in emoji.EMOJI_DATA for ch in x))
    return df

train = add_meta(train); test = add_meta(test)


In [ ]:
# 4. 可視化ライブラリをここで
!pip install japanize-matplotlib
import seaborn as sns
import matplotlib.pyplot as plt, japanize_matplotlib

cat_feats = ["has_url", "has_keyword", "has_mention",
             "has_hashtag", "has_location"]
num_feats = ["text_len", "word_cnt", "emoji_cnt"]

# -- カテゴリ特徴量：災害率 barplot ------------------
for col in cat_feats:
    plt.figure(figsize=(4,3))
    sns.barplot(x=col, y="target", data=train, errorbar=None)
    plt.title(f"{col} と災害率")
    plt.ylabel("mean(target)")
    plt.show()

# -- 数値特徴量：target ごとの分布 -------------------
for col in num_feats:
    plt.figure(figsize=(4,3))
    sns.boxplot(x="target", y=col, data=train)
    plt.title(f"{col} の分布 (target 0 vs 1)")
    plt.show()

# --- 補足: 各特徴量の災害率/平均値を一覧で確認 ------------------------
summary = pd.DataFrame({
    "feature": cat_feats + num_feats,
    "metric" : ["mean(target)"]*len(cat_feats) + ["median"]*len(num_feats),
    "value"  : [train.groupby(f)[ "target"].mean().values[1] for f in cat_feats] +
               [train.loc[train["target"]==1, f].median() for f in num_feats]
})
display(summary)

In [ ]:
# 5. 分割だけに必要な import
from sklearn.model_selection import train_test_split

TEXT_COL  = "clean_text"
BIN_COLS  = ["has_url","has_keyword","has_mention",
             "has_hashtag","has_location"]
NUM_COLS  = ["text_len","word_cnt","emoji_cnt"]
META_COLS = BIN_COLS + NUM_COLS

X = train[[TEXT_COL] + META_COLS]
y = train["target"]

X_tr, X_val, y_tr, y_val = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y)


In [ ]:
# 6. モデリング系をここで import
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

tfidf = TfidfVectorizer(max_features=20000,
                        ngram_range=(1,2),
                        stop_words="english")

pre = ColumnTransformer([
        ("tfidf", tfidf, TEXT_COL),
        ("num",   StandardScaler(), NUM_COLS),
        ("bin",   "passthrough",   BIN_COLS)
      ])

pipe = Pipeline([
        ("prep", pre),
        ("clf",  LogisticRegression(max_iter=1000,
                                    class_weight="balanced",
                                    n_jobs=-1))
      ])

pipe.fit(X_tr, y_tr)
val_pred = pipe.predict(X_val)
print("Validation F1 =", round(f1_score(y_val, val_pred), 4))


In [ ]:
# 7. 提出用 CSV を書き出すだけ
sub = pd.read_csv(f"{DATA_PATH}/sample_submission.csv")
sub["target"] = pipe.predict(test[[TEXT_COL] + META_COLS])
sub.to_csv("submission_tfidf_lr.csv", index=False)
sub.head()
